In [1]:
import os, warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection    import StratifiedKFold
from sklearn.preprocessing      import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics            import (accuracy_score, balanced_accuracy_score,
                                        f1_score, matthews_corrcoef,
                                        precision_score, recall_score,
                                        confusion_matrix, roc_auc_score,
                                        average_precision_score)
from sklearn.linear_model       import LogisticRegression
from sklearn.tree               import DecisionTreeClassifier
from sklearn.ensemble           import (RandomForestClassifier, ExtraTreesClassifier,
                                        GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.svm                import SVC
from sklearn.neural_network     import MLPClassifier
from xgboost                    import XGBClassifier
import lightgbm as lgb
from catboost                   import CatBoostClassifier

from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import (Dense, Dropout, Reshape,
                                        Conv1D, GlobalMaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")


In [2]:
# -- PATHS ---------------------------------------------------------------------
# Filenames are RELATIVE, since os.path.join with an absolute path would
# throw BASE_DIR away.
BASE_DIR   = "/kaggle/input/datasets/harshiikaaaaa/foodev-13299-plm"        # <-- update to your dataset slug
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_TRAIN = 10639         # rows in the training split; part of the filenames

#Adjust these to match the filenames you actually produced.
plm_files = {
    "ESM2-150M": f"features_esm2_150M_{N_TRAIN}.csv",
    "ESM2-650M": f"features_esm2_650M_{N_TRAIN}.csv",
    "ProtBERT" : f"features_protbert_{N_TRAIN}.csv",
    "ProtGPT2" : f"features_protgpt2_{N_TRAIN}.csv",
    "ProtT5"   : f"features_prott5_{N_TRAIN}.csv",
    "Ankh"     : f"features_ankh_{N_TRAIN}.csv",
    #"ProtT5": "/kaggle/input/datasets/harshiikkaa/foodev-v3-prott5/features_prott5_11008.csv"
}

#plm_files = {
#    "Correlation": "/kaggle/input/datasets/harshiikkaa/fev-feat-optimized/features_prott5_Correlation_6807.csv",
#    "L1": "/kaggle/input/datasets/harshiikkaa/fev-feat-optimized/features_prott5_L1_6807.csv",
#    "MI" : "/kaggle/input/datasets/harshiikkaa/fev-feat-optimized/features_prott5_MI_6807.csv",
#    "PCA" : "/kaggle/input/datasets/harshiikkaa/fev-feat-optimized/features_prott5_PCA_6807.csv",
#    "RFE"   : "/kaggle/input/datasets/harshiikkaa/fev-feat-optimized/features_prott5_RFE_6807.csv",
#    "RF"     : "/kaggle/input/datasets/harshiikkaa/fev-feat-optimized/features_prott5_RF_6807.csv",
#}

ID_COL       = "seq_id"
TARGET_COL   = "primary_label"      # 0 = Non_EV, 1 = Milk_EV, 2 = Plant_EV
STRATIFY_COL = "secondary_label"    # tens = species, units = EV/non-EV
META_COLS    = [ID_COL, "primary_label", "secondary_label"]
CLASS_NAMES  = {0: "Non_EV", 1: "Milk_EV", 2: "Plant_EV"}

N_SPLITS      = 5      # outer CV -> 80:20 fit/validation per fold
INNER_VAL_FRAC = 0.20  # inner 80:20 split for early-stopping models

# -- METRICS -------------------------------------------------------------------
def get_metrics(y_true, y_pred, y_score=None, n_classes=3):
    """
    Returns a flat dict of metrics.

      sensitivity : macro-averaged recall, TP / (TP + FN) per class, one-vs-rest
      specificity : macro-averaged TN / (TN + FP) per class, one-vs-rest
      precision   : macro-averaged TP / (TP + FP)
      npv         : macro-averaged TN / (TN + FN)
      auc / ap    : macro-averaged one-vs-rest, computed from predicted
                    probabilities. NaN for models without predict_proba.
    """
    m = {}
    m["acc"]  = accuracy_score(y_true, y_pred)
    m["bacc"] = balanced_accuracy_score(y_true, y_pred)
    m["f1"]   = f1_score(y_true, y_pred, average="macro")
    m["pre"]  = precision_score(y_true, y_pred, average="macro", zero_division=0)
    m["sens"] = recall_score(y_true, y_pred, average="macro", zero_division=0)
    m["mcc"]  = matthews_corrcoef(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    specs, npvs = [], []
    for i in range(n_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
        npvs.append(tn / (tn + fn) if (tn + fn) > 0 else 0.0)
        m[f"sens_c{i}"] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        m[f"spec_c{i}"] = specs[-1]
        m[f"pre_c{i}"]  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        m[f"npv_c{i}"]  = npvs[-1]
    m["spec"] = float(np.mean(specs))
    m["npv"]  = float(np.mean(npvs))

    if y_score is not None:
        yb = label_binarize(y_true, classes=list(range(n_classes)))
        try:
            m["auc"] = roc_auc_score(y_true, y_score, multi_class="ovr",
                                     average="macro", labels=list(range(n_classes)))
        except Exception:
            m["auc"] = np.nan
        try:
            m["ap"] = average_precision_score(yb, y_score, average="macro")
        except Exception:
            m["ap"] = np.nan
    else:
        m["auc"] = np.nan
        m["ap"]  = np.nan
    return m


def get_proba(model, X, n_classes=3):
    """
    Probabilities for AUC / AP. Returns None when the model cannot produce
    them - SVC is fitted with probability=False, and squashing decision_function
    into pseudo-probabilities would change cross-sample ranking and give a
    misleading AUC.
    """
    if hasattr(model, "predict_proba"):
        try:
            p = model.predict_proba(X)
            if p.shape[1] == n_classes:
                return p
        except Exception:
            return None
    return None


def fmt(arr, is_mcc=False):
    a = np.asarray(arr, dtype=float)
    m = 1 if is_mcc else 100
    return f"{np.nanmean(a)*m:.2f} ± {np.nanstd(a)*m:.2f}"


def fmt_ci(arr, is_mcc=False):
    """Mean with a 95% CI across folds (normal approximation on the fold SE)."""
    a = np.asarray(arr, dtype=float)
    a = a[~np.isnan(a)]
    if a.size == 0:
        return "n/a"
    m  = 1 if is_mcc else 100
    mu = a.mean() * m
    se = a.std(ddof=1) / np.sqrt(a.size) * m if a.size > 1 else 0.0
    return f"{mu:.2f} [{mu - 1.96*se:.2f}, {mu + 1.96*se:.2f}]"


In [3]:
# ── MODEL FACTORY ─────────────────────────────────────────────────────────────
# Fresh instances per PLM — avoids fitted-state bleed across feature spaces.
# Regularisation is tuned for high-dimensional embeddings (shallower trees,
# subsampling on the boosted ensembles, wider MLP hidden layers).

def get_ml_models():
    return {
        "Logistic Regression": LogisticRegression(
            C=1.0, max_iter=1000, solver="lbfgs",
            class_weight="balanced",
            random_state=42, n_jobs=-1),
            # lbfgs is multinomial by default.

        "Decision Tree": DecisionTreeClassifier(
            max_depth=5, min_samples_leaf=10,
            class_weight="balanced", random_state=42),

        "Random Forest": RandomForestClassifier(
            n_estimators=200, max_depth=8, min_samples_leaf=5,
            class_weight="balanced",
            random_state=42, n_jobs=-1),

        "Extra Trees": ExtraTreesClassifier(
            n_estimators=200, max_depth=8, min_samples_leaf=5,
            class_weight="balanced",
            random_state=42, n_jobs=-1),

        "Gradient Boosting": HistGradientBoostingClassifier(
            max_iter=200, max_depth=3, learning_rate=0.05,
            class_weight="balanced",                 # supported sklearn ≥1.2
            l2_regularization=0.1, random_state=42),

        "AdaBoost": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1),
            n_estimators=100, random_state=42),
            # No class_weight on the base stump: AdaBoost (SAMME) reweights
            # samples itself, so a stump that also re-normalises by class
            # corrupts the weighted-error estimate. Class balance is supplied
            # via sample_weight at fit time instead, as XGBoost does.

        "XGBoost": XGBClassifier(
            max_depth=4, learning_rate=0.05, n_estimators=200,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="mlogloss",
            random_state=42, n_jobs=-1, verbosity=0),
            # No class_weight param — weighted via sample_weight in fold loop

        "LightGBM": lgb.LGBMClassifier(
            max_depth=6, num_leaves=31, min_child_samples=20,
            class_weight="balanced",
            n_estimators=200, reg_lambda=0.1,
            verbosity=-1, random_state=42, n_jobs=-1),

        "CatBoost": CatBoostClassifier(
            iterations=300, depth=4, l2_leaf_reg=3,
            auto_class_weights="Balanced",
            verbose=0, random_state=42),

        "SVM (RBF)": SVC(
            kernel="rbf", C=1.0, class_weight="balanced", random_state=42),

        "SVM (Linear)": SVC(
            kernel="linear", C=1.0, class_weight="balanced", random_state=42),

        "MLP": MLPClassifier(
            hidden_layer_sizes=(512, 256, 128), alpha=0.001,
            max_iter=500, early_stopping=True,
            validation_fraction=INNER_VAL_FRAC, random_state=42),
            # inner 80:20 early-stopping split, matching DNN/CNN.
            # MLPClassifier supports neither class_weight nor sample_weight, so
            # this is the one model here trained unweighted while the others
            # are balanced.
    }

# ── DL MODELS ─────────────────────────────────────────────────────────────────
# BiGRU / BiLSTM excluded: mean-pooled PLM embeddings are a single flat
# vector with no temporal axis, so an RNN would only see one time-step.
#
# CNN : treats D embedding dims as a 1-D signal (D pseudo-timesteps x 1 channel).
#       Two conv layers learn local feature interactions across adjacent dims.
# DNN : standard feed-forward; the natural baseline for flat embeddings.

DL_MODEL_NAMES = ["DNN", "CNN"]

def build_dl_model(name, input_dim):
    model = Sequential()
    if name == "CNN":
        model.add(Reshape((input_dim, 1), input_shape=(input_dim,)))
        model.add(Conv1D(128, kernel_size=5, activation="relu", padding="same"))
        model.add(Conv1D(64,  kernel_size=3, activation="relu", padding="same"))
        model.add(GlobalMaxPooling1D())
        model.add(Dense(128, activation="relu"))
        model.add(Dropout(0.3))
    elif name == "DNN":
        model.add(Dense(512, activation="relu", input_shape=(input_dim,)))
        model.add(Dropout(0.4))
        model.add(Dense(256, activation="relu"))
        model.add(Dropout(0.3))
        model.add(Dense(128, activation="relu"))
        model.add(Dropout(0.2))
    model.add(Dense(3, activation="softmax"))
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

In [4]:
# -- LOAD LABELS FROM THE EMBEDDING FILES --------------------------------------
# Labels travel with the embeddings, so there is no second file to keep in sync.
# All PLM files are checked for identical seq_id ordering before anything is fit.

ref_meta, ref_plm = None, None
available = {}
for plm_name, fname in plm_files.items():
    path = os.path.join(BASE_DIR, fname)
    if not os.path.exists(path):
        print(f"[SKIP] {plm_name} - not found: {path}")
        continue
    meta = pd.read_csv(path, usecols=META_COLS)
    if ref_meta is None:
        ref_meta, ref_plm = meta, plm_name
    elif not meta[ID_COL].equals(ref_meta[ID_COL]):
        raise ValueError(f"{plm_name} row order differs from {ref_plm}. "
                         "All embedding files must share the same seq_id order.")
    available[plm_name] = path

if not available:
    raise FileNotFoundError(f"No embedding files found under {BASE_DIR}")

y       = ref_meta[TARGET_COL].to_numpy()
y_strat = ref_meta[STRATIFY_COL].to_numpy()
seq_ids = ref_meta[ID_COL].to_numpy()
N_CLASSES = int(len(np.unique(y)))

print("=" * 70)
print("  PLM SELECTION  |  5-Fold CV on the TRAINING SPLIT  |  14 Models")
print("=" * 70)
print(f"  PLMs available : {len(available)}  ({', '.join(available)})")
print(f"  Sequences      : {len(y)}")
print(f"  Target         : {TARGET_COL}")
for c in np.unique(y):
    print(f"      {c} = {CLASS_NAMES[int(c)]:<9} n = {(y == c).sum()}")
print("  Locked test split : NOT read by this notebook")

# Stratifying on secondary_label keeps each species proportional in every fold.
# On primary_label alone, donkey (154 train positives, 13% of Milk_EV) could
# drift between folds and add variance unrelated to the model.
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# -- SPLIT SIZES ---------------------------------------------------------------
print(f"\n  OUTER SPLIT  {N_SPLITS}-fold stratified on {STRATIFY_COL} "
      f"({len(np.unique(y_strat))} species x polarity groups)")
fold_sizes = []
for f, (tr_idx, va_idx) in enumerate(cv.split(np.zeros(len(y)), y_strat), 1):
    n_tr, n_va = len(tr_idx), len(va_idx)
    n_inner_va = int(np.floor(n_tr * INNER_VAL_FRAC))
    n_inner_tr = n_tr - n_inner_va
    fold_sizes.append(dict(fold=f, fit=n_tr, validation=n_va,
                           pct_fit=round(100*n_tr/len(y), 1),
                           inner_fit=n_inner_tr, inner_early_stop=n_inner_va))
    print(f"    fold {f}:  fit {n_tr:5d} ({100*n_tr/len(y):.1f}%)   "
          f"validation {n_va:5d} ({100*n_va/len(y):.1f}%)")
    print(f"             inner 80:20 -> fit {n_inner_tr:5d}   "
          f"early-stopping {n_inner_va:5d}")
print(f"  INNER SPLIT  {int((1-INNER_VAL_FRAC)*100)}:{int(INNER_VAL_FRAC*100)} "
      f"inside each training fold, used by DNN, CNN and MLP for early stopping.")
print("  Class balance is preserved in the outer folds by stratification and in "
      "the inner split by shuffling before the split.")
pd.DataFrame(fold_sizes).to_csv(
    os.path.join(OUTPUT_DIR, "split_sizes.csv"), index=False)

all_results = []


  PLM SELECTION  |  5-Fold CV on the TRAINING SPLIT  |  14 Models
  PLMs available : 6  (ESM2-150M, ESM2-650M, ProtBERT, ProtGPT2, ProtT5, Ankh)
  Sequences      : 10639
  Target         : primary_label
      0 = Non_EV    n = 7635
      1 = Milk_EV   n = 1340
      2 = Plant_EV  n = 1664
  Locked test split : NOT read by this notebook

  OUTER SPLIT  5-fold stratified on secondary_label (8 species x polarity groups)
    fold 1:  fit  8511 (80.0%)   validation  2128 (20.0%)
             inner 80:20 -> fit  6809   early-stopping  1702
    fold 2:  fit  8511 (80.0%)   validation  2128 (20.0%)
             inner 80:20 -> fit  6809   early-stopping  1702
    fold 3:  fit  8511 (80.0%)   validation  2128 (20.0%)
             inner 80:20 -> fit  6809   early-stopping  1702
    fold 4:  fit  8511 (80.0%)   validation  2128 (20.0%)
             inner 80:20 -> fit  6809   early-stopping  1702
    fold 5:  fit  8512 (80.0%)   validation  2127 (20.0%)
             inner 80:20 -> fit  6810   early

In [5]:
# -- MAIN LOOP -----------------------------------------------------------------
METRIC_KEYS = ["acc", "bacc", "f1", "sens", "spec", "pre", "npv", "mcc", "auc", "ap"]
PERCLASS_KEYS = [f"{s}_c{i}" for i in range(N_CLASSES)
                 for s in ("sens", "spec", "pre", "npv")]

# AdaBoost joins XGBoost here: both take class balance through sample_weight at
# fit time rather than through a class_weight parameter.
SAMPLE_WEIGHT_MODELS = {"XGBoost", "AdaBoost"}

for plm_name, file_path in available.items():

    print(f"\n{'-'*70}")
    print(f"  PLM : {plm_name}   |   {os.path.basename(file_path)}")
    print(f"{'-'*70}")

    df_feat = pd.read_csv(file_path)

    # Select features BY PREFIX. Using .values would feed seq_id and both label
    # columns into the model - i.e. train on the target.
    dim_cols = [c for c in df_feat.columns if c.startswith("dim_")]
    if not dim_cols:
        raise ValueError(f"{plm_name}: no dim_* columns. "
                         f"Saw: {list(df_feat.columns)[:8]}")
    X_raw     = df_feat[dim_cols].to_numpy(dtype=np.float32)
    input_dim = X_raw.shape[1]
    print(f"  Feature matrix : {X_raw.shape[0]} x {input_dim} "
          f"(+{len(META_COLS)} id/label cols excluded)")

    assert X_raw.shape[0] == len(y), \
        f"Row mismatch - features {X_raw.shape[0]} vs labels {len(y)}"
    assert not np.isnan(X_raw).any(), f"{plm_name}: NaNs in the embedding matrix"

    ml_models  = get_ml_models()          # fresh instances per feature space
    all_models = {**ml_models, **{n: "DL" for n in DL_MODEL_NAMES}}

    for model_name, model in all_models.items():
        print(f"  -> {model_name:<22}", end=" ", flush=True)

        tr_m = {k: [] for k in METRIC_KEYS + PERCLASS_KEYS}
        cv_m = {k: [] for k in METRIC_KEYS + PERCLASS_KEYS}

        for fold, (tr_idx, va_idx) in enumerate(cv.split(X_raw, y_strat)):

            X_tr_raw, X_va_raw = X_raw[tr_idx], X_raw[va_idx]
            y_tr,     y_va     = y[tr_idx],     y[va_idx]

            sample_w = compute_sample_weight(class_weight="balanced", y=y_tr)
            class_w  = compute_class_weight(class_weight="balanced",
                                            classes=np.unique(y_tr), y=y_tr)
            class_weight_dict = {int(c): w for c, w in zip(np.unique(y_tr), class_w)}

            scaler = StandardScaler()          # fitted on the training fold only
            X_tr   = scaler.fit_transform(X_tr_raw)
            X_va   = scaler.transform(X_va_raw)

            if model == "DL":
                # StratifiedKFold returns ascending indices and the dataset is
                # ordered by class, so an unshuffled tail split would leave
                # early stopping watching a single class. Shuffle first so
                # the inner 20% is representative.
                rs   = np.random.RandomState(42 + fold)
                perm = rs.permutation(len(y_tr))
                X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]

                curr_model = build_dl_model(model_name, input_dim)
                early_stop = EarlyStopping(monitor="val_loss", patience=5,
                                           restore_best_weights=True, verbose=0)
                curr_model.fit(
                    X_tr_s, y_tr_s,
                    epochs=60, batch_size=32, verbose=0,
                    validation_split=INNER_VAL_FRAC,       # inner 80:20
                    class_weight=class_weight_dict,
                    callbacks=[early_stop])

                s_tr = curr_model.predict(X_tr, verbose=0)
                s_va = curr_model.predict(X_va, verbose=0)
                p_tr = np.argmax(s_tr, axis=1)
                p_va = np.argmax(s_va, axis=1)
                tf.keras.backend.clear_session()

            else:
                if model_name in SAMPLE_WEIGHT_MODELS:
                    model.fit(X_tr, y_tr, sample_weight=sample_w)
                else:
                    model.fit(X_tr, y_tr)
                p_tr = model.predict(X_tr)
                p_va = model.predict(X_va)
                s_tr = get_proba(model, X_tr, N_CLASSES)
                s_va = get_proba(model, X_va, N_CLASSES)

            a = get_metrics(y_tr, p_tr, s_tr, N_CLASSES)
            b = get_metrics(y_va, p_va, s_va, N_CLASSES)
            for k in tr_m:
                tr_m[k].append(a[k])
                cv_m[k].append(b[k])

        gap = (np.mean(tr_m["acc"]) - np.mean(cv_m["acc"])) * 100
        auc_txt = ("  AUC n/a" if np.isnan(np.nanmean(cv_m["auc"]))
                   else f"  AUC {np.nanmean(cv_m['auc']):.3f}")
        print(f"| CV  ACC {np.mean(cv_m['acc'])*100:.1f}%  "
              f"SENS {np.mean(cv_m['sens'])*100:.1f}%  "
              f"SPEC {np.mean(cv_m['spec'])*100:.1f}%  "
              f"F1 {np.mean(cv_m['f1'])*100:.1f}%  "
              f"MCC {np.mean(cv_m['mcc']):.3f}{auc_txt}  gap {gap:+.1f}")

        row = {"PLM": plm_name, "Model": model_name}
        for k, lbl in [("acc","ACC"), ("bacc","BACC"), ("sens","SENS"),
                       ("spec","SPEC"), ("pre","PRE"), ("npv","NPV"),
                       ("f1","F1"), ("mcc","MCC"), ("auc","AUC"), ("ap","AP")]:
            is_mcc = k in ("mcc", "auc", "ap")
            row[f"Train_{lbl}"] = fmt(tr_m[k], is_mcc=is_mcc)
            row[f"CV_{lbl}"]    = fmt(cv_m[k], is_mcc=is_mcc)
            row[f"CV_{lbl}_95CI"] = fmt_ci(cv_m[k], is_mcc=is_mcc)
        for i in range(N_CLASSES):
            cn = CLASS_NAMES[i]
            for s, lbl in [("sens","SENS"), ("spec","SPEC"),
                           ("pre","PRE"), ("npv","NPV")]:
                row[f"CV_{lbl}_{cn}"] = fmt(cv_m[f"{s}_c{i}"])
        row["Overfit_Gap"] = round(gap, 2)
        row["_cv_f1"]  = np.mean(cv_m["f1"])
        row["_cv_mcc"] = np.mean(cv_m["mcc"])
        all_results.append(row)

# -- SAVE ----------------------------------------------------------------------
df_final = (pd.DataFrame(all_results)
              .sort_values(by=["PLM", "_cv_f1"], ascending=[True, False])
              .reset_index(drop=True))

out_path = os.path.join(OUTPUT_DIR, "PLM_14Models_5FoldCV_TRAIN.csv")
df_final.drop(columns=["_cv_f1", "_cv_mcc"]).to_csv(out_path, index=False)

print(f"\n{'='*70}")
print(f"  Saved : {out_path}   Shape : {df_final.shape}")
print(f"{'='*70}")
print(df_final[["PLM","Model","CV_ACC","CV_SENS","CV_SPEC","CV_F1","CV_MCC",
                "Overfit_Gap"]].to_string(index=False))

print(f"\n  Headline metrics with 95% CIs across folds (comment 11):")
print(df_final[["PLM","Model","CV_ACC_95CI","CV_F1_95CI","CV_MCC_95CI"]]
      .to_string(index=False))

# -- PLM RANKING ---------------------------------------------------------------
# Ranked on mean macro-F1 across all 14 models, so the chosen PLM is the one
# whose feature space is broadly strongest, not the one that pairs with a
# single lucky classifier.
print(f"\n{'='*70}")
print("  PLM RANKING  (mean across all models, CV macro-F1)")
print(f"{'='*70}")
rank = (df_final.groupby("PLM")
        .agg(mean_CV_F1=("_cv_f1", "mean"),
             best_CV_F1=("_cv_f1", "max"),
             best_model=("_cv_f1", lambda s: df_final.loc[s.idxmax(), "Model"]),
             mean_CV_MCC=("_cv_mcc", "mean"))
        .sort_values("mean_CV_F1", ascending=False).round(4))
print(rank.to_string())
rank.to_csv(os.path.join(OUTPUT_DIR, "PLM_ranking_TRAIN.csv"))
print(f"\nSelected PLM : {rank.index[0]}")
print("Carry this one PLM forward to feature selection. Everything after this "
      "point stays inside the training split; the locked test set is scored "
      "once, at the very end.")



----------------------------------------------------------------------
  PLM : ESM2-150M   |   features_esm2_150M_10639.csv
----------------------------------------------------------------------
  Feature matrix : 10639 x 640 (+3 id/label cols excluded)
  -> Logistic Regression    | CV  ACC 67.9%  SENS 68.8%  SPEC 83.0%  F1 60.8%  MCC 0.439  AUC 0.849  gap +7.8
  -> Decision Tree          | CV  ACC 53.5%  SENS 61.6%  SPEC 78.3%  F1 49.5%  MCC 0.317  AUC 0.767  gap +3.7
  -> Random Forest          | CV  ACC 69.3%  SENS 68.7%  SPEC 83.0%  F1 61.7%  MCC 0.445  AUC 0.853  gap +12.1
  -> Extra Trees            | CV  ACC 59.5%  SENS 69.1%  SPEC 81.7%  F1 55.6%  MCC 0.405  AUC 0.843  gap +7.9
  -> Gradient Boosting      | CV  ACC 70.1%  SENS 70.9%  SPEC 84.0%  F1 63.1%  MCC 0.469  AUC 0.870  gap +11.8
  -> AdaBoost               | CV  ACC 57.6%  SENS 64.6%  SPEC 79.1%  F1 53.1%  MCC 0.346  AUC 0.779  gap +1.9
  -> XGBoost                | CV  ACC 70.6%  SENS 71.2%  SPEC 84.2%  F1 63.5%  MCC 

I0000 00:00:1789673848.304047      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789673848.306957      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1789673854.419398     647 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


| CV  ACC 68.9%  SENS 69.9%  SPEC 83.7%  F1 62.0%  MCC 0.459  AUC 0.866  gap +3.7
  -> CNN                    | CV  ACC 50.9%  SENS 52.4%  SPEC 74.7%  F1 44.1%  MCC 0.218  AUC 0.712  gap +1.8

----------------------------------------------------------------------
  PLM : ESM2-650M   |   features_esm2_650M_10639.csv
----------------------------------------------------------------------
  Feature matrix : 10639 x 1280 (+3 id/label cols excluded)
  -> Logistic Regression    | CV  ACC 70.1%  SENS 64.5%  SPEC 81.4%  F1 60.4%  MCC 0.412  AUC 0.833  gap +18.0
  -> Decision Tree          | CV  ACC 52.9%  SENS 60.5%  SPEC 77.9%  F1 48.7%  MCC 0.305  AUC 0.759  gap +4.5
  -> Random Forest          | CV  ACC 69.1%  SENS 69.8%  SPEC 83.1%  F1 62.1%  MCC 0.449  AUC 0.857  gap +12.9
  -> Extra Trees            | CV  ACC 58.7%  SENS 70.0%  SPEC 81.7%  F1 55.5%  MCC 0.408  AUC 0.844  gap +8.3
  -> Gradient Boosting      | CV  ACC 72.0%  SENS 72.9%  SPEC 85.0%  F1 65.2%  MCC 0.497  AUC 0.882  gap +12.1